In [1]:
import os
import glob
import numpy as np
from nilearn.image import concat_imgs, new_img_like
from scipy.stats import ttest_1samp

# ---- Settings ----
base_dir = "/Users/liuq13/SPCD_data/searchlight_maps/odor_scan_2odors"
norm_options = ["none", "across_trials", "across_voxels"]
chance_level = 0.5

for norm in norm_options:
    print(f"\n>>> Running group analysis for normalization: {norm}")

    searchlight_dir = os.path.join(base_dir, norm)
    output_dir = os.path.join(searchlight_dir, "group_stats")
    os.makedirs(output_dir, exist_ok=True)

    map_files = sorted(glob.glob(os.path.join(searchlight_dir, "sub-*smooth6.nii.gz")))
    print(f"Found {len(map_files)} smoothed maps")

    if len(map_files) == 0:
        print("No smoothed maps found. Skipping.")
        continue

    # ---- Load into 4D image ----
    img_4d = concat_imgs(map_files)
    data_4d = img_4d.get_fdata()  # shape: (X, Y, Z, N)

    # ---- Step 2: Reshape and clean ----
    n_voxels = np.prod(data_4d.shape[:-1])
    n_subjects = data_4d.shape[-1]
    data_2d = data_4d.reshape(n_voxels, n_subjects)

    valid_mask = ~np.any(np.isnan(data_2d), axis=1)
    valid_data = data_2d[valid_mask, :]

    # ---- Step 3: Voxel-wise one-sample t-test ----
    t_vals, p_vals = ttest_1samp(valid_data - chance_level, popmean=0, axis=1, nan_policy='omit')

    # ---- Step 4: Map t-values back to brain ----
    t_map = np.full((n_voxels,), np.nan)
    t_map[valid_mask] = t_vals
    t_map_3d = t_map.reshape(data_4d.shape[:-1])
    t_img = new_img_like(img_4d.slicer[..., 0], t_map_3d)

    # ---- Step 5: Save ----
    t_img_path = os.path.join(output_dir, "group_tmap.nii.gz")
    t_img.to_filename(t_img_path)
    print(f"T-map saved to: {t_img_path}")



>>> Running group analysis for normalization: none
Found 34 smoothed maps


/Users/liuq13/Library/Python/3.9/lib/python/site-packages/scipy/stats/_axis_nan_policy.py:563: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, axis=axis, **kwds)


T-map saved to: /Users/liuq13/SPCD_data/searchlight_maps/odor_scan_2odors/none/group_stats/group_tmap.nii.gz

>>> Running group analysis for normalization: across_trials
Found 34 smoothed maps
T-map saved to: /Users/liuq13/SPCD_data/searchlight_maps/odor_scan_2odors/across_trials/group_stats/group_tmap.nii.gz

>>> Running group analysis for normalization: across_voxels
Found 34 smoothed maps
T-map saved to: /Users/liuq13/SPCD_data/searchlight_maps/odor_scan_2odors/across_voxels/group_stats/group_tmap.nii.gz
